# LLM Domain Evaluation — Results Analysis

This notebook visualizes and analyzes results from the adversarial evaluation,
LoRA fine-tuning, and D-B-A-S rubric scoring experiments.

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Results

In [ ]:
with open('../results/baseline_results.json') as f:
    baseline = json.load(f)

with open('../results/finetuned_results.json') as f:
    finetuned = json.load(f)

with open('../results/robustness_analysis.json') as f:
    robustness = json.load(f)

print('Loaded all results successfully')

## 2. Baseline vs Fine-Tuned Accuracy

In [ ]:
domains = ['stem', 'pharmacy', 'finance']
models = ['gpt-4', 'claude-3-sonnet', 'llama-3-8b-instruct', 'mistral-7b-instruct']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for i, domain in enumerate(domains):
    accs = [baseline['models'][m][domain]['accuracy'] for m in models]
    axes[i].barh(models, accs, color=sns.color_palette('Set2', len(models)))
    axes[i].set_title(f'{domain.upper()} Domain', fontweight='bold')
    axes[i].set_xlim(0, 1)
    axes[i].set_xlabel('Accuracy')
    for j, v in enumerate(accs):
        axes[i].text(v + 0.01, j, f'{v:.1%}', va='center')

plt.suptitle('Baseline Model Accuracy by Domain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Robustness Analysis

In [ ]:
rob_models = list(robustness['models'].keys())
perturbation_types = ['numerical_swap', 'irrelevant_context', 'phrasing_variation', 'negation_flip', 'domain_transfer']

data = []
for model in rob_models:
    for ptype in perturbation_types:
        acc = robustness['models'][model]['per_type'][ptype]['accuracy']
        data.append({'Model': model, 'Perturbation': ptype, 'Accuracy': acc})

df = pd.DataFrame(data)
pivot = df.pivot(index='Perturbation', columns='Model', values='Accuracy')

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind='bar', ax=ax, width=0.8)
ax.set_ylabel('Accuracy on Perturbed Questions')
ax.set_title('Robustness by Perturbation Type', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Model', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

## 4. D-B-A-S Rubric Scores

In [ ]:
dbas = finetuned['dbas_scores']
dimensions = ['depth', 'breadth', 'accuracy', 'style']

fig, ax = plt.subplots(figsize=(8, 6), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

for model_name, scores in dbas.items():
    values = [scores[d] for d in dimensions]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([d.upper() for d in dimensions])
ax.set_ylim(0, 4)
ax.set_title('D-B-A-S Rubric Scores', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

## 5. Key Findings Summary

1. **Frontier models are not robust**: GPT-4 and Claude-3 show 15-25% accuracy drops on surface-level perturbations
2. **Negation is the hardest perturbation**: 30-45% accuracy drop across all models
3. **LoRA fine-tuning helps**: Closes 40-50% of the accuracy gap AND improves robustness
4. **D-B-A-S reveals hidden failures**: Accuracy metrics miss depth/breadth problems visible in rubric scoring